# Predicting Diabetes Progression from Baseline Measurements

**Sultan Akinola Atanda** · 9114837 · CSCN8010 Foundations of Machine Learning · Conestoga College

The aim is a model that estimates disease progression one year after a baseline visit, to help physicians screen for at-risk patients. I compare univariate and multivariate polynomial regression, decision trees and kNN, then use logistic regression to flag high-risk patients directly. The code behind the analysis lives in `src/` (`DiabetesData`, `DiabetesVisualizer`, the modelling classes and `RiskDashboard`).

## 1. Getting the data

I load the data with `scaled=False` so every feature stays in clinical units (BMI in kg/m², blood pressure in mm Hg). This keeps the BMI equation readable and lets the dashboard accept real values. Scaling happens inside each model's pipeline, fitted on training data only.

In [ ]:
import pandas as pd

from src.dashboard import RiskDashboard
from src.data_pipeline import DiabetesData
from src.modeling import ClassificationEvaluator, ModelFactory, ModelInspector, RegressionEvaluator
from src.visuals import DiabetesVisualizer

pd.set_option("display.precision", 3)

data = DiabetesData(random_state=42)
df = data.load()
viz = DiabetesVisualizer(image_dir="images")
data.dictionary()

The six serum columns `s1` to `s6` were renamed to their clinical meaning.

## 2. Framing the problem

The target is a continuous score (25 to 346), so this is **supervised regression** on 442 patients with 10 features. Because the model is a screening tool, errors are not equal: missing a high-risk patient (false negative) is worse than an extra follow-up (false positive).

Points carried over from the workshops:

- **Performance Metrics:** average error can hide failure on the group that matters. I define *high risk* as the top 25% of training scores and check every model's **recall** on that group, not just its average error.
- **kNN:** predictions come from distances, so features must be standardised. Small *k* chases noise, large *k* oversmooths, and kNN learns no weights.
- **Logistic Regression / Log-Loss:** outputs a probability through the sigmoid and is trained by minimising log-loss, which punishes confident mistakes. It needs a yes/no target, which the high-risk definition provides.

Regression metrics: **R²** (share of variance explained, negative if worse than the mean), **MAE** (average miss in score points) and **MAPE** (relative miss, which inflates for patients with low scores). A degree-0 model that always predicts the mean is the baseline to beat.

## 3. Exploratory data analysis

In [ ]:
print(f"Missing values: {df.isna().sum().sum()}, duplicate rows: {data.duplicate_count()}")
data.summary()

In [ ]:
data.split()
threshold = data.risk_threshold(0.75)
viz.target_distribution(df[data.TARGET], threshold=threshold)

In [ ]:
viz.histograms(df, data.features)

In [ ]:
viz.scatter_against_target(df, data.features, data.TARGET)

In [ ]:
viz.correlation_heatmap(df)

**Insights**

- The data is complete: no missing values or duplicates, and every range is clinically plausible (BMI 18 to 42, ages 19 to 79).
- The target is right-skewed (mean 152, median 140.5) with a long tail of fast progressors. Models minimising average error will be pulled toward the middle, so the highest-risk patients will be the hardest to predict.
- **BMI (r = 0.59)** and **ltg (r = 0.57)** are the strongest predictors, followed by blood pressure (0.44) and `tch` (0.43). `hdl` is the only negative one (-0.39). Even the best scatter plot is a wide cloud, so BMI alone will leave most of the variance unexplained.
- The cholesterol features overlap heavily: `tc` and `ldl` correlate at 0.90, and `hdl` and `tch` at -0.74. This matters for polynomial models and kNN distances in Part 3.
- `sex` is a 1/2 code with almost no linear link to the target (0.04). `tch` has only 66 distinct values because the ratio was recorded coarsely.

## 4. Cleaning

No imputation or deduplication is needed. The IQR rule flags a few points per feature, so I checked the BMI ones:

In [ ]:
data.outlier_rows('bmi')

These are real patients with severe obesity and high progression, exactly the group a screener exists for, so all outliers are kept. Columns were renamed and `sex` cast to integer. Scaling is left to each pipeline to avoid leaking test data into training.

## 5. Train, validation and test split

In [ ]:
splits = data.split(train=0.75, val=0.10, test=0.15)
data.split_overview()

The validation set has only 44 patients and a lower mean target than training, so close calls on validation are double-checked with 10-fold cross-validation on the training set.

---

# Part 2: univariate polynomial regression on BMI

## 6. Models, degree 0 to 5

BMI is standardised before the polynomial expansion because raw BMI⁵ reaches about 10⁸, which makes least squares unstable. The equation in step 11 is converted back to raw BMI.

In [ ]:
bmi_splits = data.subset(["bmi"])
part2 = RegressionEvaluator(bmi_splits)
part2.fit({f"degree_{d}": ModelFactory.univariate_polynomial(d) for d in range(6)})
viz.polynomial_family(part2.models, *bmi_splits["train"])

## 7. Training vs validation comparison

In [ ]:
part2_table = part2.comparison_table()
part2_table

In [ ]:
viz.metric_comparison(part2_table, "MAE", title="BMI models: MAE on training vs validation", name="part2_mae")

Degree 1 cuts validation MAE from 66 to 47 compared with the mean baseline. Degrees 1 to 3 are tied, and degrees 4 and 5 look slightly better on validation (degree 5: MAE 45.5, R² 0.35). No model overfits badly: the R² gap stays under 0.02.

## 8. Best model

Degree 5 wins on the table, but by under two points on 44 patients. Cross-validation and a sanity check on its predictions tell a different story:

In [ ]:
grid = pd.DataFrame({"bmi": [16, 18, 20, 30, 40, 45]})
display(part2.cross_validation_table())
grid.assign(**{d: part2.models[d].predict(grid).round(0) for d in ["degree_1", "degree_5"]})

Across 10 folds degree 5 is the worst non-trivial model, and degree 1 is the best (tied with degree 2). Degree 5 also predicts higher risk at BMI 16 than at BMI 20, which makes no clinical sense. **Chosen model: degree 1.**

## 9. Test results

In [ ]:
bmi_model = part2.models["degree_1"]
part2.test_report("degree_1")

Test R² 0.28, MAE 51.5, MAPE 43.2%. The drop from training R² (0.35) matches the cross-validated estimate (0.29), so the model is limited by having one input, not by overfitting.

## 10. Fit over train, validation and test

In [ ]:
viz.fit_over_splits(bmi_model, bmi_splits)

## 11. Equation

In [ ]:
print(ModelInspector.polynomial_equation(bmi_model))

$$\text{progression} = -125.18 + 10.51 \times \text{BMI}$$

Each extra BMI unit adds about 10.5 points. Two decimals are enough here; for degrees 4 and 5 they would not be, since their high-order coefficients round to 0.00 but still matter once multiplied by BMI⁵.

## 12. Prediction for BMI = 32

In [ ]:
print(f"BMI 32 -> {bmi_model.predict(pd.DataFrame({'bmi': [32.0]}))[0]:.1f} (high-risk cut-off: {threshold:.0f})")

A clearly obese patient lands at 211, just *below* the high-risk line. A first sign that BMI alone is a weak screener.

## 13. Trainable parameters

In [ ]:
pd.DataFrame(
    {
        "terms": [", ".join(m.named_steps["poly"].get_feature_names_out()) for m in part2.models.values()],
        "parameters": [ModelInspector.parameter_count(m) for m in part2.models.values()],
    },
    index=part2.models.keys(),
)

A degree-*d* model fits **d + 1** coefficients, one per power of BMI from 0 to *d*. The `1` column from `PolynomialFeatures` carries the intercept, so the regression uses `fit_intercept=False` to avoid counting it twice. The scaler's mean and standard deviation are computed, not trained, so they are not counted.

## 14. Part 2 conclusions

In [ ]:
X_test_bmi, y_test = bmi_splits["test"]
viz.actual_vs_predicted(y_test, bmi_model.predict(X_test_bmi), "BMI model: actual vs predicted (test)",
                        "part2_actual_vs_predicted", threshold=threshold)

In [ ]:
display(part2.error_by_range("degree_1"))
part2.screening_table(threshold, names=["degree_1"], split="test")

**Performance.** The BMI line beats the mean baseline (test MAE 51.5 vs 66) but explains only 28% of the variance.

**Where it fails**

- It squeezes predictions toward the middle: the slowest progressors are over-predicted (bias +51, MAPE 86%) and the fastest under-predicted (bias -66, MAE 75).
- As a screener it catches only **5 of 19** high-risk test patients (recall 0.26). This is the Performance Metrics lesson: decent average error, poor recall on the class that matters.

**Limitations.** One feature cannot separate patients with the same BMI but different blood pressure or triglycerides; the line assumes a constant BMI effect; 442 patients from one study; MAPE exaggerates errors on low scores; and the model shows association, not cause.

---

# Part 3: multivariate models

## 15. Features and models

I drop `tc` and `ldl`: they duplicate each other (r = 0.90), are the weakest cholesterol signals (0.17 and 0.21), and their information is largely kept in `tch` and `hdl`. `sex` stays because it can still matter in interactions. A plain linear model is included as a reference.

In [ ]:
selected = [f for f in data.features if f not in ("tc", "ldl")]
multi_splits = data.subset(selected)

part3 = RegressionEvaluator(multi_splits)
part3.fit(
    {
        "poly_degree_2": ModelFactory.multivariate_polynomial(2),
        "poly_degree_3": ModelFactory.multivariate_polynomial(3),
        "tree_depth_3": ModelFactory.decision_tree(max_depth=3),
        "tree_depth_5": ModelFactory.decision_tree(max_depth=5, min_samples_leaf=5),
        "knn_k5": ModelFactory.knn(5),
        "knn_k11": ModelFactory.knn(11),
        "linear_reference": ModelFactory.multivariate_polynomial(1),
    }
)
display(part3.comparison_table())
part3.cross_validation_table()

## 16. Comparison and best model

- **Degree-3 polynomial** overfits badly (train R² 0.71, validation -0.14) with 165 coefficients for 331 patients.
- **Trees** fit training well but generalise poorly (R² gaps of 0.31 and 0.34); hard splits suit a gradual relationship badly.
- **kNN, k = 11** has the best validation R² (0.45), near-best MAE (40.9) and the smallest gap (0.05). Raising *k* from 5 to 11 traded a little training fit for stability.
- The **linear reference** has the best cross-validated MAE (46.0). Extra flexibility is not buying much with 442 patients.

**Chosen model: kNN with k = 11.**

In [ ]:
knn_model = part3.models["knn_k11"]
part3.test_report("knn_k11")

Test R² **0.49**, MAE **42.8**, MAPE **36.3%**. Compared with BMI alone, R² nearly doubles and MAE drops by 9 points.

## 17. Trainable parameters

In [ ]:
pd.DataFrame(
    {
        "parameters": [ModelInspector.parameter_count(m) for m in part3.models.values()],
        "detail": [ModelInspector.parameter_breakdown(m) if "poly" not in n and "linear" not in n else ""
                   for n, m in part3.models.items()],
    },
    index=part3.models.keys(),
)

- **Polynomials:** 8 features give 44 degree-2 terms (8 linear, 8 squares, 28 interactions) plus an intercept = 45, and 164 + 1 = 165 at degree 3. The linear model has 8 weights + intercept = 9.
- **Trees:** every internal node learns a threshold and every leaf a value, so the count equals the node count (7 + 8 = 15 and 25 + 26 = 51).
- **kNN: 0.** It stores the 331 training rows and does all the work at prediction time.

## 18. Logistic regression for high-risk screening

High risk = score ≥ 214 (75th percentile of training targets, computed on training data only). About a quarter of patients are positive, so I compare a default model against one with `class_weight="balanced"` and stronger regularisation. R², MAE and MAPE do not apply to class labels, so classification metrics are used.

In [ ]:
classifier = ClassificationEvaluator(multi_splits, threshold)
classifier.fit(
    {
        "logistic_default": ModelFactory.logistic(C=1.0),
        "logistic_balanced": ModelFactory.logistic(C=0.1, class_weight="balanced"),
    }
)
classifier.comparison_table()

Both rank patients equally well (validation ROC-AUC 0.93 to 0.94), but the default model catches only 3 of 7 high-risk validation patients, while the balanced one catches 6 of 7. The default model has lower log-loss, a reminder that the metric being optimised is not always the one that fits the use case. **Chosen: `logistic_balanced`.**

In [ ]:
display(classifier.test_report("logistic_balanced"))
viz.confusion_matrix(classifier.confusion("logistic_balanced"), "Balanced logistic regression (test)",
                     "logistic_confusion_matrix")

On the test set it catches **14 of 19** high-risk patients (recall 0.74, ROC-AUC 0.88). For comparison, the regression models with the same cut-off applied to their scores:

In [ ]:
part3.screening_table(threshold, split='test')

## 19. Part 3 conclusions

In [ ]:
X_test, y_test = multi_splits["test"]
viz.actual_vs_predicted(y_test, knn_model.predict(X_test), "kNN (k = 11): actual vs predicted (test)",
                        "part3_actual_vs_predicted", threshold=threshold)

In [ ]:
part3.error_by_range('knn_k11')

**Where kNN fails.** The middle of the range is now predicted well (MAE 23 for scores 92 to 150), but the extremes are still pulled inward (bias +50 at the bottom, -54 at the top). Averaging 11 neighbours can never produce a value more extreme than the patients it averages, so it flags only 6 of 19 high-risk patients, against 14 for the logistic model.

**Limitations.** kNN cannot extrapolate beyond the training range and has to ship with patient data; the 214 cut-off is a modelling choice rather than a clinical definition; the 44-patient validation set makes close rankings ties; and the sample is small and from a single study.

## Overall conclusions

1. More features help (test R² 0.28 to 0.49); more flexibility does not, as the degree-3 polynomial and deep tree show.
2. The validation set alone picked a degree-5 polynomial that cross-validation rejected.
3. Every regression model under-predicts the fastest progressors.
4. **For screening, classify directly:** the balanced logistic model catches 74% of high-risk patients against 32% for the best regressor.

Next steps would be a regularised linear model (ridge or lasso), tuning the classification threshold with clinicians, and validating on a separate patient group.

*References: Efron et al. (2004), Least Angle Regression, Annals of Statistics 32(2); scikit-learn `load_diabetes` documentation.*

---

# Screening dashboard

Move the sliders, pick a score model and press **Predict**. The score comes from the chosen regressor and the risk probability from the balanced logistic model.

In [ ]:
RiskDashboard(
    regressors={
        "kNN, k = 11": knn_model,
        "Linear (8 features)": part3.models["linear_reference"],
        "BMI only": bmi_model,
    },
    classifier=classifier.models["logistic_balanced"],
    X_train=multi_splits["train"][0],
    threshold=threshold,
).show()